# Retraining 2CaP-5

## Read in inputs
### Read in libraries

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras import backend as K
from ML_helper_functions import PearsonCorr, OneHot, OneHotSeries, preprocess_RNA, train_full_model, pearson_correlation, spearman_correlation, MSE, MAE, Transformer, PositionalEmbedding, preprocess_RNA_embedding, RNA_tokenizer

2026-01-29 15:39:30.175460: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-29 15:39:30.197427: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-29 15:39:30.197454: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-29 15:39:30.198036: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-29 15:39:30.202361: I tensorflow/core/platform/cpu_feature_guar

### Read in the model

In [2]:
filepath = "RESULT/MODEL_D5/model_d5.tf"

# Load in the model #
objects = {"OneHotSeries":  OneHotSeries, "PearsonCorr": PearsonCorr}    
model = tf.keras.models.load_model(filepath, custom_objects=objects)

np.random.seed(42)

2026-01-29 15:39:31.378465: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-29 15:39:31.418155: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-29 15:39:31.418197: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-29 15:39:31.419558: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-29 15:39:31.419589: I external/local_xla/xla/stream_executor

### Read in the data

You read your training data in here. As an example we use the dataset from [1].
Ideally training and testing data are separate datasets (even different experiments). If such a dataset is not at hand you can split your data into a training and testing dataset (eg. 85% training and 15% testing).

[1] Kim, H. K., Min, S., Song, M., Jung, S., Choi, J. W., Kim, Y., ... & Kim, H. H. (2018). Deep learning improves prediction of CRISPR–Cpf1 guide RNA activity. Nature biotechnology, 36(3), 239-241.

In [3]:
# training dataset #    
deepcpf1_train = pd.read_excel('DeepCpf1.xlsx', sheet_name='Data set HT 1-1', skiprows=1)
deepcpf1_train = deepcpf1_train.loc[:,["34 bp synthetic target and target context sequence\n(4 bp + PAM + 23 bp protospacer + 3 bp)", "Indel freqeuncy\n(Cpf1 delivered,%)"]]

deepcpf1_train.columns = ['crRNA', 'efficiency']
deepcpf1_train.dropna(axis=0, inplace=True)
deepcpf1_train = deepcpf1_train.sample(frac=1, random_state=42)
X = preprocess_RNA(deepcpf1_train.loc[:,'crRNA'].to_numpy())
y = deepcpf1_train.loc[:,'efficiency'].to_numpy()/100.0

deepcpf1_train.head()

2026-01-29 15:39:33.692196: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


,crRNA,efficiency
11499,CTACTTTTACGGGGTGCCTTTTGGACTTTTAAGC,0.000000
6475,GCCCTTTGAAACGCCACACAGCCGAGGCGATGCG,15.905512
13167,ATGATTTTGAAGTTGTTCAATTTTCTCCAGCCCT,8.400810
862,CAGTTTTAGGGGAAGATCTAGGTGTGTGACGTCT,0.485437
5970,GTTTTTTCTTCCCCAGCTGTCTCTGCTGCTGTCC,58.042829


In [4]:
# testing dataset # 
deepcpf1_test = pd.read_excel('DeepCpf1.xlsx', sheet_name='Data set HT 1-2', skiprows=1)
deepcpf1_test = deepcpf1_test.loc[:,["34 bp synthetic target and target context sequence\n(4 bp + PAM + 23 bp protospacer + 3 bp)", "Indel freqeuncy\n(Cpf1 delivered,%)"]]

deepcpf1_test.columns = ['crRNA', 'efficiency']
deepcpf1_test.dropna(axis=0, inplace=True)
X_test = preprocess_RNA(deepcpf1_test.loc[:,'crRNA'].to_numpy())
y_test = deepcpf1_test.loc[:,'efficiency'].to_numpy()/100.0

deepcpf1_test.head()

,crRNA,efficiency
0,TTGCTTTAAAACTCGCAAGGCTTTCTGCTTGACC,73.775671
1,CAGATTTAAAGACTTTCTGCTGTATTTGAGATGC,68.350168
2,GTAGTTTAAAGCCTTTTTTATTGTATCTTGTTGC,0.343643
3,CCCTTTTAAATTCTGTGCAGACCATAGTGCTGCT,74.155405
4,TTACTTTAACACTCTCAGTTGGCCCATATTCACA,43.073341


In [5]:
n_val = int(0.15*X.shape[0])

X_val = X[:n_val,:]
y_val = y[:n_val]

X_train = X[n_val:,:]
y_train = y[n_val:]

print('Training dataset:')
print('X_train: ' + str(X_train.shape))
print('y_train: ' + str(y_train.shape))
print('Validation dataset:')
print('X_val: ' + str(X_val.shape))
print('y_val: ' + str(y_val.shape))
print('Testing dataset:')
print('X_test: ' + str(X_test.shape))
print('y_test: ' + str(y_test.shape))

Training dataset:
X_train: (12750, 34)
y_train: (12750,)
Validation dataset:
X_val: (2250, 34)
y_val: (2250,)
Testing dataset:
X_test: (1292, 34)
y_test: (1292,)


## Performance

In [6]:
# Get the untrained performance # 

model = tf.keras.models.load_model(filepath, custom_objects=objects)
y_pred = model.predict(X_test, verbose=0)
y_pred = y_pred.reshape((-1,))
        
result_mse = MSE(y_test, y_pred)
result_pearson = pearson_correlation(y_test, y_pred)
result_spearman = spearman_correlation(y_test, y_pred)

print('Untrained performance of 2CaP-5:')
print('Pearson: ' + str(result_pearson))
print('Spearman: ' + str(result_spearman))

2026-01-29 15:39:34.703953: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2026-01-29 15:39:34.760560: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Untrained performance of 2CaP-5:
Pearson: 0.27010565648202023
Spearman: 0.25315917134959354


In [7]:
# Get the trained performance # 

model = tf.keras.models.load_model(filepath, custom_objects=objects)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True)
model.fit(X_train, y_train, validation_data=(X_val, y_val), batch_size=32,
              epochs=200, verbose=1, callbacks=[early_stopping_callback])

y_pred = model.predict(X_test, verbose=1)
y_pred = y_pred.reshape((-1,))
        
result_mse = MSE(y_test, y_pred)
result_pearson = pearson_correlation(y_test, y_pred)
result_spearman = spearman_correlation(y_test, y_pred)

print('Trained performance of 2CaP-5:')
print('Pearson: ' + str(result_pearson))
print('Spearman: ' + str(result_spearman))

Epoch 1/200


2026-01-29 15:39:36.540837: I external/local_xla/xla/service/service.cc:168] XLA service 0x79c1c1697540 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-01-29 15:39:36.540866: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3090, Compute Capability 8.6
2026-01-29 15:39:36.544000: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1769697576.580269  548914 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


399/399 [==============================] - 4s 6ms/step - loss: 0.0831 - mae: 0.2440 - pearson_corr: 0.4505 - val_loss: 0.0618 - val_mae: 0.2061 - val_pearson_corr: 0.6736
Epoch 2/200
399/399 [==============================] - 2s 5ms/step - loss: 0.0654 - mae: 0.2076 - pearson_corr: 0.6144 - val_loss: 0.0515 - val_mae: 0.1820 - val_pearson_corr: 0.7183
Epoch 3/200
399/399 [==============================] - 2s 5ms/step - loss: 0.0590 - mae: 0.1942 - pearson_corr: 0.6619 - val_loss: 0.0544 - val_mae: 0.1850 - val_pearson_corr: 0.7191
Epoch 4/200
399/399 [==============================] - 2s 5ms/step - loss: 0.0564 - mae: 0.1886 - pearson_corr: 0.6834 - val_loss: 0.0494 - val_mae: 0.1749 - val_pearson_corr: 0.7300
Epoch 5/200
399/399 [==============================] - 2s 5ms/step - loss: 0.0545 - mae: 0.1838 - pearson_corr: 0.6938 - val_loss: 0.0481 - val_mae: 0.1728 - val_pearson_corr: 0.7386
Epoch 6/200
399/399 [==============================] - 2s 5ms/step - loss: 0.0530 - mae: 0.1817 -

In [8]:
tf.keras.models.save_model(model, 'retrained2CaP_5.tf')

INFO:tensorflow:Assets written to: retrained2CaP_5.tf/assets


INFO:tensorflow:Assets written to: retrained2CaP_5.tf/assets
